In [ ]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson
from fooof import FOOOF
from fooof.sim.gen import gen_aperiodic
from fooof.plts.spectra import plot_spectra
from fooof.plts.annotate import plot_annotated_peak_search
from fooof import FOOOFGroup

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_shape_st = (60, 900)

fit foof for each subject for each channel separately but across trials.
In the end you arrive at an exponent and offset value for each trial per channel per subject.
THen correlate the offset and the exponent with the predicted amplitude?

In [ ]:
def extract_fractal(data, min_peak_height=0.05, max_n_peaks=6, peak_width_limits=[2.2, 8],aperiodic_mode='fixed'):
    psd, freqs = mne.time_frequency.psd_array_multitaper(data, 1000, fmin=2, fmax=45, adaptive=True, low_bias=True, normalization='full', verbose=False, bandwidth=10)
    aperiodoc_exponent = []
    aperiodoc_offset = []
    errors = []
    r2s = []
    for ch_idx in range(data.shape[1]):
        fg = FOOOFGroup(peak_width_limits=peak_width_limits, min_peak_height=min_peak_height, max_n_peaks=max_n_peaks, aperiodic_mode=aperiodic_mode)
        fg.fit(freqs, psd[:,ch_idx], [2, 45])

        aperiodic_exponent_channel = fg.get_params("aperiodic_params", "exponent")
        aperiodic_offset_channel = fg.get_params("aperiodic_params", "offset")
        aperiodoc_exponent.append(aperiodic_exponent_channel)
        aperiodoc_offset.append(aperiodic_offset_channel)

        errors_mean_channel = np.mean(fg.get_params("error"))
        r2s_mean_channel = np.mean(fg.get_params("r_squared"))
        errors.append(errors_mean_channel)
        r2s.append(r2s_mean_channel)

    return aperiodoc_exponent, aperiodoc_offset, errors, r2s

In [ ]:
from scipy.stats import pearsonr
def plot_fractal_component_amplitude(component_data, amplitude_data, ch_names, subject_index=2, show_plot=True, corr_threshold=0.35):
    # for each channel make a plot with the component data on the x axis the amplitude on the y axis 
    # with a separate datapoint for each trial
    if show_plot:

        fig, axs = plt.subplots(nrows=len(ch_names)//4, ncols=4, figsize=(20, 40), gridspec_kw={'hspace': 0.6, 'wspace': 0.15}, sharey=True)
        fig.suptitle(f"Subject {subject_index} Fractal Component vs Amplitude", fontsize=16)
        fig.tight_layout()
    
    high_corr_channels = {ch_name: 0 for ch_name in ch_names}
    corrs_abs = {}
    corrs = {}
    for ch_idx, ch_name in enumerate(ch_names):
        corrs[ch_name] = {}
        corrs_abs[ch_name] = {}
        high_corr_channels[ch_name] = {}
        comp_data = component_data[ch_idx]
        amp_data = amplitude_data
        
        # Calculate thresholds
        comp_low, comp_high = np.percentile(comp_data, [1, 99])
        amp_low, amp_high = np.percentile(amp_data, [1, 99])
        
        # Filter data
        valid_indices = (comp_data > comp_low) & (comp_data < comp_high) & (amp_data > amp_low) & (amp_data < amp_high)
        filtered_comp_data = comp_data[valid_indices]
        filtered_amp_data = amp_data[valid_indices]
        
        corr, pval = pearsonr(filtered_comp_data, filtered_amp_data)
        if np.abs(corr) >= corr_threshold:
            high_corr_channels[ch_name] = corr
        
        corrs[ch_name]["stat"] = corr
        corrs_abs[ch_name]["stat"] = np.abs(corr)
        corrs[ch_name]["pval"] = pval
        corrs_abs[ch_name]["pval"] = pval
        if show_plot:
            if np.abs(corr) >= 0.35:
                axs[ch_idx//4, ch_idx%4].scatter(filtered_comp_data, filtered_amp_data, color="black", alpha=0.4)
                
            else:
                axs[ch_idx//4, ch_idx%4].scatter(filtered_comp_data, filtered_amp_data, alpha=0.4)
            axs[ch_idx//4, ch_idx%4].set_xlabel("Fractal Component")
            axs[ch_idx//4, ch_idx%4].set_ylabel("Amplitude")
            axs[ch_idx//4, ch_idx%4].set_title(f"{ch_name}, corr = {corr:.2f}")

    return high_corr_channels, corrs, corrs_abs
        

# subject 2 analysis

In [ ]:
save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/data"
file_path = os.path.join(save_path, "subject_002_preprocessed_combined_py.fif")
data = mne.read_epochs(file_path)
epochs = data.get_data()[150:,:,:900]
cfg = load_config()
cfg.dataset.subject_index = 2
all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)#all_epochs = all_epochs[150:,:,:900]

In [ ]:
#aperiodoc_exponent, aperiodoc_offset, errors, r2s = extract_fractal(all_epochs)

In [ ]:
#aperiodoc_exponent = np.array(aperiodoc_exponent)
#aperiodoc_offset = np.array(aperiodoc_offset)

In [ ]:
#all_subject_amplitude_data = np.load(os.path.join(save_path, "all_subject_amplitude_data.npy"), allow_pickle=True).item()

In [ ]:
#exponents_results = plot_fractal_component_amplitude(aperiodoc_exponent, all_subject_amplitude_data[2], ch_names, show_plot=True)

In [ ]:
#exponents_results_clean = {k: v for k, v in exponents_results.items() if v!=0}

In [ ]:
#exponents_results_clean

In [ ]:
#offset_results = plot_fractal_component_amplitude(aperiodoc_offset, all_subject_amplitude_data[2], ch_names)

# all subjects

#dont execute
import pickle
all_subject_amplitude_data = {}
all_subject_uncertainty_data = {}
data_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject in cfg.dataset.test_subject_indices:
    with open(os.path.join(data_path, f"subject_{subject}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        all_subject_amplitude_data[subject] = data['predictions']
        all_subject_uncertainty_data[subject] = data['uncertainties']

cwd = os.getcwd()
all_subjects_high_corr_channels = {}
for subject_index in cfg.dataset.test_subject_indices:
    file_path = os.path.join(save_path, f"subject_{subject_index:03d}_preprocessed_combined_py.fif")
    
    cfg.dataset.subject_index = subject_index

    all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)
    all_epochs = all_epochs[150:,:,:900]
    aperiodoc_exponent, aperiodoc_offset, errors, r2s = extract_fractal(all_epochs)
    subejct_dict = {"exponent": aperiodoc_exponent, "offset": aperiodoc_offset, "errors": errors, "r2s": r2s}
    print(f"Subject {subject_index}")
    print(f"mean error = {np.mean(errors)}")
    print(f"mean r2 = {np.mean(r2s)}")
    save_dir = os.path.join(cwd, f"subject_{subject_index}")
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, f"fractal_results_subject_{subject_index}.npy"), subejct_dict)
    
    all_subjects_high_corr_channels[subject_index] = plot_fractal_component_amplitude(aperiodoc_exponent, all_subject_amplitude_data[subject_index], ch_names, subject_index=subject_index, show_plot=False)
    
   
    

#save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/frequency_power_data"
cfg = load_config()

cfg.exp_name = f"S4_S4EEGNet_ema_100_cal_py_{cfg.dataset.subject_index}"
cli_args = parse_args()
cfg = update_config(cfg, cli_args)
save_config(cfg)
all_epochs, all_labels_raw, fixed_median, fixed_q1, fixed_q3, labels_scaler, mean_mep, ch_names = load_eeg_data(cfg)

In [ ]:
import pickle
#cfg = load_config()
dir_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/frequency_power_analysis/frequency_power_data"
cfg = load_config()
all_subject_power_data_integral = np.load(os.path.join(dir_path, "all_subject_power_data_integral.npy"), allow_pickle=True).item()
all_subject_power_data_average = np.load(os.path.join(dir_path, "all_subject_power_data_average.npy"), allow_pickle=True).item()
#all_subject_amplitude_data = np.load(os.path.join(dir_path, "all_subject_amplitude_data.npy"), allow_pickle=True).item()
#all_subject_uncertainty_data = np.load(os.path.join(dir_path, "all_subject_uncertainty_data.npy"), allow_pickle=True).item()
all_subject_amplitude_data = {}
all_subject_uncertainty_data = {}
data_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject in cfg.dataset.test_subject_indices:
    with open(os.path.join(data_path, f"subject_{subject}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        all_subject_amplitude_data[subject] = data['predictions']
        all_subject_uncertainty_data[subject] = data['uncertainties']

In [ ]:
all_subjects_highest_corrs_all_exponent = {}
all_subjects_highest_corrs_all_abs_exponent = {}
all_subjects_high_corr_channels_exponent = {}
all_subjects_highest_corrs_all_offset = {}
all_subjects_highest_corrs_all_abs_offset = {}
all_subjects_high_corr_channels_offset = {}

cfg = load_config()
cwd = os.getcwd()
all_subjects_high_corr_channels = {}
for subject_index in cfg.dataset.test_subject_indices:
   dir = np.load(os.path.join(cwd, f"subject_{subject_index}/fractal_results_subject_{subject_index}.npy"), allow_pickle=True).item()
   aperiodoc_exponent, aperiodoc_offset, errors, r2s = dir["exponent"], dir["offset"], dir["errors"], dir["r2s"] 
   all_subjects_high_corr_channels_exponent[subject_index],all_subjects_highest_corrs_all_exponent[subject_index], all_subjects_highest_corrs_all_abs_exponent[subject_index] = plot_fractal_component_amplitude(aperiodoc_exponent, all_subject_amplitude_data[subject_index], ch_names, subject_index=subject_index, show_plot=False)

   all_subjects_high_corr_channels_offset[subject_index],all_subjects_highest_corrs_all_offset[subject_index], all_subjects_highest_corrs_all_abs_offset[subject_index] = plot_fractal_component_amplitude(aperiodoc_offset, all_subject_amplitude_data[subject_index], ch_names, subject_index=subject_index, show_plot=False)
    
   
    

In [ ]:
#np.save('highest_corrs_fractal.npy', all_subjects_highest_corrs_all_abs)

In [ ]:
#np.save(os.path.join(cwd, "all_subjects_highest_corrs_all_aperiodic_exponent.npy"), all_subjects_highest_corrs_all)
#np.save(os.path.join(cwd, "all_subjects_highest_corrs_all_abs_aperiodic_exponent.npy"), all_subjects_highest_corrs_all_abs)

In [ ]:
#np.save(os.path.join(cwd, "all_subjects_highest_corrs_all_aperiodic_offset.npy"), all_subjects_highest_corrs_all)
#np.save(os.path.join(cwd, "all_subjects_highest_corrs_all_abs_aperiodic_offset.npy"), all_subjects_highest_corrs_all_abs)

In [ ]:
# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 10-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(aggregate_dict, aggregate_dict_key, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict


cwd = os.getcwd()
dir_path = os.path.join(cwd, "frequency_power_data")

freq_bands = {"theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}



In [ ]:
def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict)
    return channel_points_dict

def get_top_k_weighted(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return top_k_channels, channel_point_dicts_over_time
    

def get_top_k_weighted_individual(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]
    #top_k_channels_dict = {ch: sum_over_channel_points_dict[ch] for ch in top_k_channels}
    

    return top_k_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict


def top_k_groupby(aggregate_dict, aggregate_dict_key, groupby_dict, ch_names, k, update_func, top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices)
    
    return groupby_channel_points_dict
 

In [ ]:
cfg = load_config()
top_10_per_subject = []
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_10_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap)}, "gradshap", ch_names, 10, update_channel_points_linear)
    top_10_per_subject.append(top_10_channels)
    

In [ ]:

all_subjects_high_corr_channels_exponent_clean = {k: {k2: v2 for k2, v2 in v.items() if v2!=0} for k, v in all_subjects_high_corr_channels_exponent.items()}


all_subjects_high_corr_channels_offset_clean = {k: {k2: v2 for k2, v2 in v.items() if v2!=0} for k, v in all_subjects_high_corr_channels_offset.items()}

#all_subjects_high_corr_channels_offset_clean = {k: {k2: v2 for k2, v2 in v.items() if v2!=0} for k, v in all_subject_highest_corrs_channels_offset.items()}

In [ ]:
def compare_channels(cfg, top_k_per_subject, all_subjects_highest_corrs, ax=None):
    subject_ratios = []
    subject_common_channels = []
    subject_sig_channels = []
    for subject_index in cfg.dataset.test_subject_indices:
        print(f"Subject {subject_index}:")
        print("Top k channels:", top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)])
        print("channels with highest correlation:", all_subjects_highest_corrs[subject_index].keys())
        
        common_channels = set(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]) & set(all_subjects_highest_corrs[subject_index].keys())
        num_common_channels = len(common_channels)
        num_significant_channels = len(all_subjects_highest_corrs[subject_index].keys())
        denominator = max(1, min(len(top_k_per_subject[cfg.dataset.test_subject_indices.index(subject_index)]), len(all_subjects_highest_corrs[subject_index].keys())))
        ratio = num_common_channels / denominator
        subject_common_channels.append(num_common_channels)
        subject_ratios.append(ratio)
        subject_sig_channels.append(num_significant_channels)
        print(f"number of common channels: {num_common_channels}")
        print(f"ratio: {ratio:.2f}")
    
    mean_agreement = np.mean(subject_ratios)
    std_agreement = np.std(subject_ratios)
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 5))
    
    ax.bar(np.arange(len(subject_ratios)), subject_ratios)
    for idx, num_sig in enumerate(subject_sig_channels):
        if num_sig == 0:
            ax.plot(idx, subject_ratios[idx], 'ro')

    ax.axhline(y=0.3, color='red', linestyle='--')
    ax.set_xticks(np.arange(len(cfg.dataset.test_subject_indices)))
    ax.set_xticklabels(np.arange(len(cfg.dataset.test_subject_indices)), rotation=45)
    ax.set_xlabel('subject index')
    ax.set_ylabel('ratio of common channels')
    ax.set_title('common channels fractal exponent')
    
    if ax is None:
        plt.savefig(f"common_channels_ratio_fractal_exponent_top_10.png")
        plt.show()
    
    return mean_agreement, std_agreement




In [ ]:
# Example usage:
fig, ax = plt.subplots(figsize=(10, 5))
mean_agreement, std_agreement = compare_channels(cfg, top_10_per_subject, all_subjects_high_corr_channels_exponent_clean, ax=ax)
print(f"Mean Agreement: {mean_agreement:.2f}")
print(f"Standard Deviation: {std_agreement:.2f}")


In [ ]:
# Example usage:
fig, ax = plt.subplots(figsize=(10, 5))
mean_agreement, std_agreement = compare_channels(cfg, top_10_per_subject, all_subjects_high_corr_channels_offset_clean, ax=ax)
print(f"Mean Agreement: {mean_agreement:.2f}")
print(f"Standard Deviation: {std_agreement:.2f}")

In [ ]:
def compare_channels_dual(cfg, top_k_per_subject, exponent_corrs, offset_corrs, figsize=(20, 8), ax=None, save_path=None):
    # Calculate ratios and agreements for both exponent and offset
    exponent_ratios = []
    offset_ratios = []
    exponent_sig_channels = []
    offset_sig_channels = []
    
    for i, subject_index in enumerate(cfg.dataset.test_subject_indices):
        # Exponent calculations
        common_channels_exponent = set(top_k_per_subject[i]) & set(exponent_corrs[subject_index].keys())
        num_common_exponent = len(common_channels_exponent)
        num_sig_exponent = len(exponent_corrs[subject_index].keys())
        denominator_exponent = max(1, min(len(top_k_per_subject[i]), num_sig_exponent))
        ratio_exponent = num_common_exponent / denominator_exponent
        
        # Offset calculations
        common_channels_offset = set(top_k_per_subject[i]) & set(offset_corrs[subject_index].keys())
        num_common_offset = len(common_channels_offset)
        num_sig_offset = len(offset_corrs[subject_index].keys())
        denominator_offset = max(1, min(len(top_k_per_subject[i]), num_sig_offset))
        ratio_offset = num_common_offset / denominator_offset
        
        exponent_ratios.append(ratio_exponent)
        offset_ratios.append(ratio_offset)
        exponent_sig_channels.append(num_sig_exponent)
        offset_sig_channels.append(num_sig_offset)
    
    # Calculate mean and std for both
    mean_agreement_exponent = np.mean(exponent_ratios)
    std_agreement_exponent = np.std(exponent_ratios)
    mean_agreement_offset = np.mean(offset_ratios)
    std_agreement_offset = np.std(offset_ratios)
    
    # Create the plot if ax not provided
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    
    # Set positions for bars
    x = np.arange(len(cfg.dataset.test_subject_indices))
    width = 0.35
    
    # Plot bars
    rects1 = ax.bar(x - width/2, exponent_ratios, width, label='Fractal Exponent', color='cornflowerblue')
    rects2 = ax.bar(x + width/2, offset_ratios, width, label='Fractal Offset', color='lightcoral')
    
    # Add a horizontal line for threshold
    ax.axhline(y=0.4, color='red', linestyle='--', linewidth=2, label='Threshold (0.4)')
    
    # Mark subjects with zero significant channels
    for i, (num_exp, num_off) in enumerate(zip(exponent_sig_channels, offset_sig_channels)):
        if num_exp == 0:
            ax.plot(i - width/2, exponent_ratios[i], 'ko', markersize=8)
        if num_off == 0:
            ax.plot(i + width/2, offset_ratios[i], 'ko', markersize=8)
    
    # Customize plot
    ax.set_xlabel('Subject Index', fontsize=18)
    ax.set_ylabel('Ratio of Common Channels', fontsize=18)
    ax.set_title('Top 10 Agreement Between Model Explanations and Fractal Components', fontsize=20, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([str(idx) for idx in cfg.dataset.test_subject_indices], rotation=45, fontsize=15)
    ax.tick_params(axis='y', labelsize=18)
    ax.legend(fontsize=15)
    
    # Add text with mean and std
    #exponent_text = f'Exponent - Mean: {mean_agreement_exponent:.2f}, Std: {std_agreement_exponent:.2f}'
    #offset_text = f'Offset - Mean: {mean_agreement_offset:.2f}, Std: {std_agreement_offset:.2f}'
    #ax.annotate(exponent_text, xy=(0.01, 0.96), xycoords='axes fraction', fontsize=14)
    #ax.annotate(offset_text, xy=(0.01, 0.92), xycoords='axes fraction', fontsize=14)
    
    # Add grid
    ax.grid(axis='y', alpha=0.3)
    
    # Only apply tight_layout if creating a new figure
    if ax is None:
        plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    return (mean_agreement_exponent, std_agreement_exponent), (mean_agreement_offset, std_agreement_offset)


In [ ]:
compare_channels_dual(cfg, top_10_per_subject, all_subjects_high_corr_channels_exponent_clean, all_subjects_high_corr_channels_offset_clean, figsize=(20, 8), save_path="common_channels_ratio_fractal_exponent_offset_top_10.png")

In [ ]:
#compare_channels(cfg, top_k_per_subject, all_subjects_high_corr_channels_offset_clean)

# compare top 10 channels without correlation cutoff

In [ ]:
all_subjects_highest_corrs_all_exponent = {}
all_subjects_highest_corrs_all_abs_exponent = {}
all_subjects_high_corr_channels_exponent = {}
all_subjects_high_corr_channels_offset = {}
all_subjects_highest_corrs_all_offset = {}
all_subjects_highest_corrs_all_abs_offset = {}


cfg = load_config()
cwd = os.getcwd()
all_subjects_high_corr_channels = {}
for subject_index in cfg.dataset.test_subject_indices:
   dir = np.load(os.path.join(cwd, f"subject_{subject_index}/fractal_results_subject_{subject_index}.npy"), allow_pickle=True).item()
   aperiodoc_exponent, aperiodoc_offset, errors, r2s = dir["exponent"], dir["offset"], dir["errors"], dir["r2s"] 
   
   all_subjects_high_corr_channels_exponent[subject_index],all_subjects_highest_corrs_all_exponent[subject_index], all_subjects_highest_corrs_all_abs_exponent[subject_index] = plot_fractal_component_amplitude(aperiodoc_exponent, all_subject_amplitude_data[subject_index], ch_names, subject_index=subject_index, show_plot=False, corr_threshold=-1)

   all_subjects_high_corr_channels_offset[subject_index],all_subjects_highest_corrs_all_offset[subject_index], all_subjects_highest_corrs_all_abs_offset[subject_index] = plot_fractal_component_amplitude(aperiodoc_offset, all_subject_amplitude_data[subject_index], ch_names, subject_index=subject_index, show_plot=False, corr_threshold=-1)

    
   

In [ ]:
np.save('highest_corrs_fractal_exponent.npy', all_subjects_highest_corrs_all_exponent)
np.save('highest_corrs_fractal_offset.npy', all_subjects_highest_corrs_all_offset)

In [ ]:
np.save('highest_corrs_abs_fractal_exponent.npy', all_subjects_highest_corrs_all_abs_exponent)
np.save('highest_corrs_abs_fractal_offset.npy', all_subjects_highest_corrs_all_abs_offset)

In [ ]:
cfg = load_config()
top_60_per_subject = []
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_60_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap)}, "gradshap", ch_names, 60, update_channel_points_linear)
    top_60_per_subject.append(top_60_channels)

In [ ]:
def extract_top_10_channels_and_mean_corr(all_subjects_corrs, take_abs=False, k=60):
    top_10_channels_per_subject = {}
    mean_corr_per_subject = {}
    
    for subject, channels in all_subjects_corrs.items():
        if take_abs:
            sorted_channels = sorted(channels.items(), key=lambda item: abs(item[1]), reverse=True)[:k]
        else:
            sorted_channels = sorted(channels.items(), key=lambda item: item[1], reverse=True)[:k]
        top_10_channels_per_subject[subject] = dict(sorted_channels)

        if take_abs:
            mean_corr = np.mean([abs(corr) for ch, corr in sorted_channels])
        else:
            mean_corr = np.mean([corr for ch, corr in sorted_channels])
        mean_corr_per_subject[subject] = mean_corr
    
    return top_10_channels_per_subject, mean_corr_per_subject

top_10_channels_per_subject_exponet, mean_corr_per_subject_exponent = extract_top_10_channels_and_mean_corr(all_subjects_highest_corrs_all_abs_exponent)

In [ ]:
all_subjects_highest_corrs_all_exponent
all_subjects_highest_corrs_all_offset

In [ ]:
top_10_channels_per_subject_exponent, mean_corr_per_subject_exponent = extract_top_10_channels_and_mean_corr(all_subjects_highest_corrs_all_exponent)

top_10_channels_per_subject_offset, mean_corr_per_subject_offset = extract_top_10_channels_and_mean_corr(all_subjects_highest_corrs_all_offset)

In [ ]:
def compute_summary_top_10(top_10):
    # across subjects and channels compute the min value, max value, the mean of the absolute values and the standard deviation
    all_corrs = []
    all_corrs_abs = []
    for subject, channels in top_10.items():
            for channel, value in channels.items():
                all_corrs.append(value)
                all_corrs_abs.append(abs(value))
    summary = {
        "min": np.min(all_corrs),
        "max": np.max(all_corrs),
        "mean": np.mean(all_corrs_abs),
        "std": np.std(all_corrs_abs)
    }
  
    return summary

In [ ]:
compute_summary_top_10(top_10_channels_per_subject_exponent)

In [ ]:
compute_summary_top_10(top_10_channels_per_subject_offset)

In [ ]:
compare_channels_dual(cfg, top_10_per_subject, top_10_channels_per_subject_exponent,top_10_channels_per_subject_offset)

# compute rank correlations for all 60 channels

In [ ]:
cfg = load_config()
top_60_per_subject = []
top_60_per_subject_dict_abs = []
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_60_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": np.abs(gradshap)}, "gradshap", ch_names, 10, update_channel_points_linear)
    top_60_per_subject.append(top_60_channels)
    top_60_per_subject_dict_abs.append(sum_over_channel_points_dict)

In [ ]:
cfg = load_config()
top_60_per_subject = []
top_60_per_subject_dict = []
load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
for subject_index in cfg.dataset.test_subject_indices:
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)


    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    top_60_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict = get_top_k_weighted_individual({"gradshap": gradshap}, "gradshap", ch_names, 10, update_channel_points_linear)
    top_60_per_subject.append(top_60_channels)
    top_60_per_subject_dict.append(sum_over_channel_points_dict)

In [ ]:
top_60_per_subject_dict = np.load("all_subject_channel_importances_gradshap_abs.npy", allow_pickle=True).item()
top_60_per_subject_dict_abs = np.load("all_subject_channel_importances_gradshap_abs.npy", allow_pickle=True).item()

In [ ]:
from scipy.stats import spearmanr

def compute_rank_correlations(top_60_per_subject_dict, all_subject_highest_corrs_all_abs):
    correlations = {}
    pvals = {}
    for subject_index in top_60_per_subject_dict.keys():
        print(subject_index)
        rank_corr, pval = spearmanr(list(top_60_per_subject_dict[subject_index].values()), list(all_subject_highest_corrs_all_abs[subject_index].values()))
        print(f"Subject {subject_index}: Spearman Rank Correlation: {rank_corr}, p-value: {pval}")
        correlations[subject_index] = rank_corr
        pvals[subject_index] = pval
    return correlations,pvals

# Compute rank correlations
rank_correlations_abs_exponent,pvals_abs_exponent = compute_rank_correlations(top_60_per_subject_dict_abs, all_subjects_highest_corrs_all_abs_exponent)
print(rank_correlations_abs_exponent)

In [ ]:
rank_correlations_abs_offset,pvals_abs_offset = compute_rank_correlations(top_60_per_subject_dict_abs, all_subjects_highest_corrs_all_abs_offset)

In [ ]:
from scipy.stats import spearmanr

def compute_rank_correlations(top_60_per_subject_dict, all_subject_highest_corrs_all_abs):
    correlations = {}
    pvals = {}
    for subject_index in top_60_per_subject_dict.keys():
        rank_corr, pval = spearmanr(list(top_60_per_subject_dict[subject_index].values()), list(all_subject_highest_corrs_all_abs[subject_index].values()))
        print(f"Subject {subject_index}: Spearman Rank Correlation: {rank_corr}, p-value: {pval}")
        correlations[subject_index] = rank_corr
        pvals[subject_index] = pval
    return correlations,pvals

# Compute rank correlations
rank_correlations_exponent,pvals_exponent= compute_rank_correlations(top_60_per_subject_dict, all_subjects_highest_corrs_all_exponent)


In [ ]:
from scipy.stats import spearmanr

def compute_rank_correlations(top_60_per_subject_dict, all_subject_highest_corrs_all_abs):
    correlations = {}
    pvals = {}
    for subject_index in top_60_per_subject_dict.keys():
        rank_corr, pval = spearmanr(list(top_60_per_subject_dict[subject_index].values()), list(all_subject_highest_corrs_all_abs[subject_index].values()))
        print(f"Subject {subject_index}: Spearman Rank Correlation: {rank_corr}, p-value: {pval}")
        correlations[subject_index] = rank_corr
        pvals[subject_index] = pval
    return correlations,pvals

# Compute rank correlations
rank_correlations_exponent,pvals_exponent= compute_rank_correlations(top_60_per_subject_dict, all_subjects_highest_corrs_all_exponent)


In [ ]:
def compute_pairwise_rank_corrs(dict1):
    all_corrs = []
    for subject1 in cfg.dataset.test_subject_indices:
        for subject2 in cfg.dataset.test_subject_indices:
            if subject2 >= subject1:
                continue
            else:
                stats1 = [dict1[subject1][ch]["stat"] for ch in dict1[subject1].keys()]
                stats2 = [dict1[subject2][ch]["stat"] for ch in dict1[subject2].keys()]
                rank_corr, pval = spearmanr(stats1, stats2)
                all_corrs.append(rank_corr)
    
    # Return mean correlation across all pairs
    return np.mean(all_corrs), np.std(all_corrs)


In [ ]:
compute_pairwise_rank_corrs(all_subjects_highest_corrs_all_exponent)

In [ ]:
compute_pairwise_rank_corrs(all_subjects_highest_corrs_all_offset)

In [ ]:
all_subjects_highest_corrs_all_offset

In [ ]:
compute_pairwise_rank_correlations(all_subjects_highest_corrs_all_offset)

In [ ]:
rank_correlations_offset,pvals_offset= compute_rank_correlations(top_60_per_subject_dict, all_subjects_highest_corrs_all_offset)

In [ ]:
def plot_rank_correlations(cfg, exponent_corrs, offset_corrs, exponent_pvals, offset_pvals, significance=0.05, figsize=(20, 8), save_path=None, ax=None):
    """
    Plot rank correlations between model explanations and fractal components (exponent and offset).
    
    Parameters:
    -----------
    cfg : config object
        Configuration object containing subject indices
    exponent_corrs : dict
        Dictionary mapping subject IDs to exponent correlations
    offset_corrs : dict
        Dictionary mapping subject IDs to offset correlations
    exponent_pvals : dict
        Dictionary mapping subject IDs to exponent p-values
    offset_pvals : dict
        Dictionary mapping subject IDs to offset p-values
    significance : float
        Significance threshold for p-values (default: 0.05)
    figsize : tuple
        Figure size (width, height) if creating a new figure
    save_path : str or None
        Path to save the figure, if None the figure will not be saved
    ax : matplotlib.axes.Axes or None
        Existing axes to plot on, if None a new figure will be created
    
    Returns:
    --------
    tuple
        ((mean_exponent, std_exponent), (mean_offset, std_offset))
    """
    
    # Create new figure if no axis is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        
    # Prepare data for plotting
    subjects = cfg.dataset.test_subject_indices
    exponent_values = [exponent_corrs[subj] for subj in subjects]
    offset_values = [offset_corrs[subj] for subj in subjects]
    
    # Calculate statistics
    mean_exponent = np.mean(list(exponent_corrs.values()))
    std_exponent = np.std(list(exponent_corrs.values()))
    mean_offset = np.mean(list(offset_corrs.values()))
    std_offset = np.std(list(offset_corrs.values()))
    
    # Set positions for bars
    x = np.arange(len(subjects))
    width = 0.35
    
    # Plot bars
    rects1 = ax.bar(x - width/2, exponent_values, width, label='Fractal Exponent', color='cornflowerblue')
    rects2 = ax.bar(x + width/2, offset_values, width, label='Fractal Offset', color='lightcoral')
    
    # Mark significant correlations
    for i, subj in enumerate(subjects):
        if exponent_pvals[subj] < significance:
            ax.plot(i - width/2, exponent_values[i], 'k*', markersize=10)
        if offset_pvals[subj] < significance:
            ax.plot(i + width/2, offset_values[i], 'k*', markersize=10)
    
    # Add zero line
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=1)
    
    # Customize plot
    ax.set_xlabel('Subject Index', fontsize=18)
    ax.set_ylabel('Spearman Rank Correlation', fontsize=18)
    ax.set_title('Rank Correlations Between Explanation function and Fractal Components', fontsize=20, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([str(idx) for idx in subjects], rotation=45, fontsize=15)
    ax.tick_params(axis='y', labelsize=18)
    ax.tick_params(axis='x', labelsize=18)
    ax.legend(fontsize=16, loc=(0,0))
    
    # Add text with mean and std
    #exponent_text = f'Exponent - Mean: {mean_exponent:.2f}, Std: {std_exponent:.2f}'
    #offset_text = f'Offset - Mean: {mean_offset:.2f}, Std: {std_offset:.2f}'

    
    # Add grid
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    return (mean_exponent, std_exponent), (mean_offset, std_offset)

In [ ]:
def plot_only_rank_correlations(cfg, exponent_corrs, offset_corrs, exponent_pvals, offset_pvals, significance=0.05, figsize=(16, 5), save_path=None, ax=None):
    """
    Plot only rank correlations between model explanations and fractal components (exponent and offset).
    
    Parameters:
    -----------
    cfg : config object
        Configuration object containing subject indices
    exponent_corrs : dict
        Dictionary mapping subject IDs to exponent correlations
    offset_corrs : dict
        Dictionary mapping subject IDs to offset correlations
    exponent_pvals : dict
        Dictionary mapping subject IDs to exponent p-values
    offset_pvals : dict
        Dictionary mapping subject IDs to offset p-values
    significance : float
        Significance threshold for p-values (default: 0.05)
    figsize : tuple
        Figure size (width, height)
    save_path : str or None
        Path to save the figure, if None the figure will not be saved
    ax : matplotlib.axes.Axes or None
        Existing axes to plot on, if None a new figure will be created
    
    Returns:
    --------
    tuple
        ((mean_exponent, std_exponent), (mean_offset, std_offset))
    """
    # Create figure and axis if not provided
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    
    # Use the existing plot_rank_correlations function with the provided axis
    (mean_exponent, std_exponent), (mean_offset, std_offset) = plot_rank_correlations(
        cfg, exponent_corrs, offset_corrs, exponent_pvals, 
        offset_pvals, significance, figsize, None, ax
    )
    
    # Only apply tight_layout if we created a new figure
    if ax is None:
        plt.tight_layout()
    
    if save_path:
        # If ax was provided, we need to get its figure
        fig = ax.figure
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    
    return (mean_exponent, std_exponent), (mean_offset, std_offset)

# Example usage:
fig, ax = plt.subplots(figsize=(16, 5))
stats = plot_only_rank_correlations(cfg, rank_correlations_exponent, rank_correlations_offset, 
                                  pvals_exponent, pvals_offset, significance=0.05/2, 
                                  ax=ax)


In [ ]:
stats_abs = plot_only_rank_correlations(cfg, rank_correlations_abs_exponent, rank_correlations_abs_offset, pvals_exponent, pvals_offset, save_path="rank_correlations_fractal_exponent_offset_abs.png")

In [ ]:
rank_correlations_abs_exponent

In [ ]:
import seaborn as sns


def plot_rank_correlations_histogram(cfg, rank_correlations_exponent, rank_correlations_offset, 
                                     rank_correlations_abs_exponent, rank_correlations_abs_offset, 
                                     pvals_exponent=None, pvals_offset=None, 
                                     pvals_abs_exponent=None, pvals_abs_offset=None,
                                     significance_level=(0.05/2),
                                     figsize=(16, 10), save_path=None, ax=None):
    """
    Plot histograms of rank correlations between model explanations and fractal components.
    """
    
    # Create figure and axes
    fig, axs = plt.subplots(2, 2, figsize=figsize, sharey=True, sharex=True)
    
    # Convert dictionaries to lists for plotting
    subjects = list(cfg.dataset.test_subject_indices)
    exponent_values = [rank_correlations_exponent[subj] for subj in subjects]
    offset_values = [rank_correlations_offset[subj] for subj in subjects]
    abs_exponent_values = [rank_correlations_abs_exponent[subj] for subj in subjects]
    abs_offset_values = [rank_correlations_abs_offset[subj] for subj in subjects]
    
    # Count significant correlations if p-values are provided
    sig_count_exponent = 0
    sig_count_offset = 0
    sig_count_abs_exponent = 0
    sig_count_abs_offset = 0
    
    if pvals_exponent is not None:
        sig_count_exponent = sum(1 for subj in subjects if pvals_exponent[subj] < significance_level)
    if pvals_offset is not None:
        sig_count_offset = sum(1 for subj in subjects if pvals_offset[subj] < significance_level)
    if pvals_abs_exponent is not None:
        sig_count_abs_exponent = sum(1 for subj in subjects if pvals_abs_exponent[subj] < significance_level)
    if pvals_abs_offset is not None:
        sig_count_abs_offset = sum(1 for subj in subjects if pvals_abs_offset[subj] < significance_level)
    
    # Calculate statistics
    mean_exponent = np.mean(exponent_values)
    std_exponent = np.std(exponent_values)
    mean_offset = np.mean(offset_values)
    std_offset = np.std(offset_values)
    mean_abs_exponent = np.mean(abs_exponent_values)
    std_abs_exponent = np.std(abs_exponent_values)
    mean_abs_offset = np.mean(abs_offset_values)
    std_abs_offset = np.std(abs_offset_values)
    
    # Define a function to plot each histogram
    def plot_hist(ax, values, color, title, mean, std, sig_count, total_subjects):
        sns.histplot(values, kde=True, ax=ax, color=color, edgecolor=color, alpha=0.8)
        ax.axvline(mean, color='black', linestyle='-', linewidth=1.5)
        ax.grid(axis='y', alpha=0.7)
        ax.grid(axis='x', visible=False)
        ax.set_title(title, fontsize=16, fontweight='bold')
        ax.set_xlabel('Spearman Rank Correlation', fontsize=16, fontweight='bold')
        ax.set_xlim([-0.5, 0.5])
        ax.text(0.05, 0.92, f"Mean: {mean:.2f}", transform=ax.transAxes,
               fontsize=14, bbox=dict(facecolor='white', alpha=0.8))
        # Add text showing significant correlations
        ax.text(0.05, 0.82, f"Significant: {sig_count}/{total_subjects}", 
                transform=ax.transAxes, fontsize=16, 
                bbox=dict(facecolor='white', alpha=0.8))
        ax.tick_params(axis='x', labelsize=18)
        ax.tick_params(axis='y', labelsize=18)
    
    # Number of subjects
    total_subjects = len(subjects)
    
    # Plot all four histograms with new colors
    plot_hist(axs[0, 0], exponent_values, 'royalblue', 'Fractal Exponent', 
             mean_exponent, std_exponent, sig_count_exponent, total_subjects)
    plot_hist(axs[0, 1], offset_values, 'indianred', 'Fractal Offset', 
             mean_offset, std_offset, sig_count_offset, total_subjects)
    plot_hist(axs[1, 0], abs_exponent_values, 'royalblue', 'Absolute Fractal Exponent', 
             mean_abs_exponent, std_abs_exponent, sig_count_abs_exponent, total_subjects)
    plot_hist(axs[1, 1], abs_offset_values, 'indianred', 'Absolute Fractal Offset', 
             mean_abs_offset, std_abs_offset, sig_count_abs_offset, total_subjects)
    
    # Add a main title
    fig.suptitle('Distribution of Rank Correlations Between Explanation Function and Fractal Components', 
                fontsize=16, y=1.02, fontweight='bold') 
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')


In [ ]:
import matplotlib.pylab as pylab
params = {'legend.fontsize': 'x-large',
          'figure.titlesize': 'x-large',
          'figure.figsize': (15, 5),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

In [ ]:
plot_rank_correlations_histogram(cfg, rank_correlations_exponent, rank_correlations_offset, rank_correlations_abs_exponent, rank_correlations_abs_offset, pvals_exponent=pvals_exponent, pvals_offset=pvals_offset, pvals_abs_exponent=pvals_abs_offset, pvals_abs_offset=pvals_abs_offset ,save_path="rank_correlations_fractal_histograms.png")

In [ ]:
print("Stats:")
print(f"Exponent: Mean = {stats[0][0]:.4f}, Std = {stats[0][1]:.4f}")
print(f"Offset: Mean = {stats[1][0]:.4f}, Std = {stats[1][1]:.4f}")

print("\nStats (absolute values):")
print(f"Exponent: Mean = {stats_abs[0][0]:.4f}, Std = {stats_abs[0][1]:.4f}")
print(f"Offset: Mean = {stats_abs[1][0]:.4f}, Std = {stats_abs[1][1]:.4f}")

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(14,8), sharex=True)
plot_only_rank_correlations(cfg, rank_correlations_abs_exponent, rank_correlations_abs_offset, pvals_exponent, pvals_offset, save_path=None, ax=axs[0])
axs[0].set_title('Rank Correlations Between Explanation function and Fractal Components', fontsize=20, fontweight='bold')
plot_only_rank_correlations(cfg, rank_correlations_exponent, rank_correlations_offset, pvals_exponent, pvals_offset, save_path=None, ax=axs[1])
axs[1].set_title('Rank Correlations Between Explanation function and Fractal Components (Absolute)', fontsize=20, fontweight='bold')
axs[1].legend().remove()

plt.tight_layout()
plt.savefig("rank_correlations_fractal_exponent_offset.png", dpi=300, bbox_inches='tight')


In [ ]:
pairwise_rank_corrs_abs, _= compute_rank_correlations(all_subjects_highest_corrs_all_abs_exponent, all_subjects_highest_corrs_all_abs_offset)
pairwise_rank_corrs,_ = compute_rank_correlations(all_subjects_highest_corrs_all_exponent, all_subjects_highest_corrs_all_offset)


In [ ]:
stats_pairwise_abs= np.mean(list(pairwise_rank_corrs_abs.values())), np.std(list(pairwise_rank_corrs_abs.values()))

In [ ]:
stats_pairwise = np.mean(list(pairwise_rank_corrs.values())), np.std(list(pairwise_rank_corrs.values()))

In [ ]:
# Print statistics for rank correlations between exponent and offset components
print("Pairwise rank correlations between exponent and offset components:")
print(f"Mean = {stats_pairwise[0]:.4f}, Std = {stats_pairwise[1]:.4f}")
print("\nPairwise rank correlations between absolute exponent and offset components:")
print(f"Mean = {stats_pairwise_abs[0]:.4f}, Std = {stats_pairwise_abs[1]:.4f}")